In [1]:
library(vctrs)
library(readr)
library(dplyr)

Warning message:
"package 'vctrs' was built under R version 4.4.3"
Warning message:
"package 'readr' was built under R version 4.4.3"
Warning message:
"package 'dplyr' was built under R version 4.4.3"

Attaching package: 'dplyr'


The following object is masked from 'package:vctrs':

    data_frame


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union




In [3]:
#低维数据预处理
pima_data <- read_csv("diabetes.csv")
cols_to_fix <- c("Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI")
pima_clean <- pima_data %>%
mutate(across(all_of(cols_to_fix), ~na_if(., 0))) %>%
mutate(across(all_of(cols_to_fix), ~ifelse(is.na(.), median(., na.rm = TRUE), .)))
y_pima <- pima_clean$Outcome
X_pima <- pima_clean %>% select(-Outcome)
X_pima_std <- scale(X_pima)
X_pima_final <- cbind(Intercept = 1, X_pima_std)
print(dim(X_pima_final))

Rows: 768 Columns: 9
── Column specification ────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
dbl (9): Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, D...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


[1] 768   9


In [4]:
#中维数据预处理：分层抽样
creditcard_data <- read_csv("creditcard_2023.csv")
set.seed(42)
creditcard_normal <- creditcard_data %>% filter(Class == 0) %>% sample_n(300)
creditcard_fraud <- creditcard_data %>% filter(Class == 1) %>% sample_n(300)
cc_data <- bind_rows(creditcard_normal, creditcard_fraud)
y_cc <- cc_data$Class
X_cc <- cc_data %>% select(-Class, -id)
X_cc_std <- scale(X_cc)
X_cc_final <- cbind(Intercept = 1, X_cc_std)
print(dim(X_cc_final))

Rows: 568630 Columns: 31
── Column specification ────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
dbl (31): id, V1, V2, V3, V4, V5, V6, V7, V8, V9, V10, V11, V12, V13, V14, V...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


[1] 600  30


In [6]:
#高维数据预处理：主成分降维
tcga_features <- read_csv("data.csv")
tcga_labels <- read_csv("labels.csv")
X_tcga <- tcga_features %>% select(-1)
y_tcga <- ifelse(tcga_labels$Class == "BRCA", 1, 0)
X_tcga_clean <- X_tcga %>% select(where(~ var(., na.rm = TRUE) > 0))
pca_result <- prcomp(X_tcga_clean, center = TRUE, scale. = TRUE)
X_tcga_pca <- pca_result$x[, 1:60]
X_tcga_std <- scale(X_tcga_pca)
X_tcga_final <- cbind(Intercept = 1, X_tcga_std)
print(paste("剔除的零方差基因数量：", ncol(X_tcga) - ncol(X_tcga_clean)))
print(paste("参与降维的有效基因数量：", ncol(X_tcga_clean)))
print(dim(X_tcga_final))
print(length(y_tcga))

New names:
• `` -> `...1`
Rows: 801 Columns: 20532
── Column specification ────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr     (1): ...1
dbl (20531): gene_0, gene_1, gene_2, gene_3, gene_4, gene_5, gene_6, gene_7,...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
New names:
• `` -> `...1`
Rows: 801 Columns: 2
── Column specification ────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr (2): ...1, Class

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


[1] "剔除的零方差基因数量： 267"
[1] "参与降维的有效基因数量： 20264"
[1] 801  61
[1] 801


In [7]:
pima_data_new <- as.data.frame(cbind(y = y_pima, X_pima_final))
write_csv(pima_data_new, "pima_preprocessed.csv")
cc_data_new <- as.data.frame(cbind(y = y_cc, X_cc_final))
write_csv(cc_data_new, "creditcard_preprocessed.csv")
tcga_data_new <- as.data.frame(cbind(y = y_tcga, X_tcga_final))
write_csv(tcga_data_new, "tcga_preprocessed.csv")